# Food Delivery Time Prediction — Complete Project (Parts 1 + 3 + 4 merged)

**Goal:** predict `Time_taken(min)` for a food delivery order.

This single notebook merges the three separate notebooks of the project:

| Original notebook | What it covered |
|---|---|
| `Project_1Part_1_.ipynb` | Data loading, cleaning, missing-value handling, first feature engineering (order hour, haversine distance) |
| `Project_1_Part_3_.ipynb` | Everything from Part 1 + EDA on features, outlier removal, encoding, scaling, and the 4 models |
| `Project_1_Part_4.ipynb` | Final version — single CSV source, extra `order_period` experiment, final model comparison + feature importances |

**Pipeline order in this notebook:**

1. Imports
2. Load data
3. First look at the data (EDA)
4. Data cleaning
5. Missing value treatment
6. Feature engineering (order hour + haversine distance)
7. Feature selection
8. Outlier handling + per-feature EDA
9. Encoding categorical features
10. Train/test split + scaling
11. Model training (Linear Regression, Decision Tree, Random Forest, XGBoost)
12. Results comparison + conclusion

---
## 1. Imports

Same import block used at the top of all three notebooks. Model-specific imports
(sklearn / xgboost) are done in Section 11 where they are used, exactly like the
original notebooks, so each model block stays self-contained.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

---
## 2. Load the data

Two different data sources were used across the notebooks:

* **Part 1 / Part 3** → separate `train.csv` and `test.csv` from
  `/content/drive/MyDrive/Food delivery dataset/`
* **Part 4 (final)** → one combined file
  `/content/drive/MyDrive/ml datasets/food delivery .csv`

Part 4 is the version this merged notebook follows, because the whole modelling
pipeline in Part 4 only ever uses `train_df` (the split into train/test is done
later with `train_test_split`). The Part 1/3 loading code is kept just below,
commented out, so nothing is lost.

In [ ]:
# ---- FINAL VERSION (Part 4): one combined CSV ----
train_df = pd.read_csv("/content/drive/MyDrive/ml datasets/food delivery .csv")

# ---- EARLIER VERSION (Part 1 & Part 3): separate train/test files ----
# test_df  = pd.read_csv("/content/drive/MyDrive/Food delivery dataset/test.csv")
# train_df = pd.read_csv("/content/drive/MyDrive/Food delivery dataset/train.csv")

---
## 3. First look at the data

Basic sanity checks — shape, dtypes, and a peek at the rows. Note in `.info()`
that almost every column comes in as `object`, even the numeric ones
(`Delivery_person_Age`, `Delivery_person_Ratings`, `multiple_deliveries`) —
that is what Section 4 fixes.

In [ ]:
train_df.head(10)

In [ ]:
train_df.shape

In [ ]:
# Everything is object dtype -> numeric columns are stored as strings,
# and missing values are the literal string "NaN" instead of np.nan.
train_df.info()

In [ ]:
train_df.multiple_deliveries.value_counts()

In [ ]:
train_df['Road_traffic_density'].value_counts()

In [ ]:
# Part 1 / Part 3 only: quick check on the separate test file
# test_df.info()

---
## 4. Data cleaning

Three problems to fix:

1. **String columns** carry leading/trailing spaces and the literal text `"NaN"`
   instead of a real `np.nan`.
2. **`Time_taken(min)`** is stored as `"(min) 24"` → need the number only.
3. **`Weatherconditions`** is stored as `"conditions Sunny"` → need the word only.

###
1 nan string ko replace krte hai actual np.nan
2. remove leading and trailing spaces

*(original comment kept from Part 1 / Part 3 / Part 4)*

In [ ]:
train_df.select_dtypes(include="object").columns

In [ ]:
# Strip whitespace on every object column, then turn the string "NaN" into a real np.nan
for col in train_df.select_dtypes(include="object").columns:
  train_df[col]=train_df[col].str.strip()
  train_df[col]=train_df[col].replace('NaN',np.nan)

In [ ]:
train_df.head()

### 4.1 Extracting the target `Time_taken(min)`

The value looks like `"(min) 24"`, so we split on the space and take index 1.
The two scratch cells below are from the original notebook — they were used to
prove that the `pd.notnull()` guard is needed, otherwise `np.nan.split()` blows up.

In [ ]:
train_df['Time_taken(min)']

In [ ]:
# scratch check: what happens on a missing value?
temp=np.nan

In [ ]:
int(temp.split(" ")[1]) if pd.notnull(temp) else temp

In [ ]:
# "(min) 24" -> 24
train_df['Time_taken(min)']=train_df["Time_taken(min)"].apply(
    lambda temp:int(temp.split(" ")[1]) if pd.notnull(temp) else temp
)

### 4.2 Cleaning `Weatherconditions`

`"conditions Sunny"` -> `"Sunny"`. Same split trick, same null guard.

In [ ]:
train_df['Weatherconditions']=train_df["Weatherconditions"].apply(
    lambda temp:temp.split(" ")[1] if pd.notnull(temp) else temp
)

# Part 1 also applied the same fix to the separate test file:
# test_df['Weatherconditions']=test_df["Weatherconditions"].apply(
#     lambda temp:temp.split(" ")[1] if pd.notnull(temp) else temp
# )

### 4.3 Converting string columns to numeric

`errors='coerce'` turns anything unparseable into `NaN`, which the next section
fills in.

In [ ]:
train_df.info()

In [ ]:
for col in ['Delivery_person_Age','Delivery_person_Ratings','multiple_deliveries'] :
  train_df[col] = pd.to_numeric(train_df[col], errors='coerce')

# Part 1 / Part 3 also converted the same columns in test_df:
# for col in ['Delivery_person_Age','Delivery_person_Ratings','multiple_deliveries'] :
#   test_df[col] = pd.to_numeric(test_df[col], errors='coerce')

In [ ]:
# Dtypes are correct now — numeric columns show as float64
train_df.info()

---
## 5. Missing value treatment

Simple, robust strategy:

* **Numeric columns → median** (median instead of mean because ratings/age are skewed
  and median is not pulled by outliers)
* **Categorical columns → mode** (most frequent category)

In [ ]:
train_df['Road_traffic_density'].mode()[0]

In [ ]:
# Numeric -> median
for col in ['Delivery_person_Age','Delivery_person_Ratings','multiple_deliveries','Vehicle_condition'] :
  train_df[col].fillna(train_df[col].median(),inplace=True)

In [ ]:
# Categorical -> mode
for col in ['City','Festival','Road_traffic_density','Weatherconditions'] :
  train_df[col].fillna(train_df[col].mode()[0],inplace=True)

In [ ]:
# No nulls left in the columns we care about
train_df.info()

In [ ]:
train_df.head()

---
## 6. Feature engineering

Two brand-new features are created from raw columns:

1. **`order_hour`** — the hour of day the order was placed, pulled out of `Time_Orderd`.
   Lunch/dinner rush hours behave very differently from off-peak.
2. **`distance_km`** — great-circle (haversine) distance between the restaurant and
   the delivery location. The raw lat/long columns are useless to a model on their
   own; the *distance between them* is the real signal.

### 6.1 `order_hour` from `Time_Orderd`

In [ ]:
# Parse the order time into a real datetime so we can pull the hour out of it
train_df['order_time_object']=pd.to_datetime(train_df['Time_Orderd'],format='%H:%M:%S', errors='coerce')

In [ ]:
train_df['order_hour']=train_df['order_time_object'].dt.hour

In [ ]:
# Rows where Time_Orderd was missing/unparseable -> fill with median hour
train_df['order_hour'].fillna(train_df['order_hour'].median(),inplace=True)

In [ ]:
train_df.head()

In [ ]:
# Helper column no longer needed after extracting the hour
train_df.drop(columns=['order_time_object'],inplace=True)

In [ ]:
train_df.head()

### 6.2 `distance_km` — vectorised haversine distance

In [ ]:
def calculate_series_distance(lat1_series, lon1_series, lat2_series, lon2_series, miles=False):
    """
    Calculates the great-circle distance between two series of points
    using the vectorised Haversine formula on NumPy arrays.
    """
    # Earth's radius: 6371 km or 3956 miles
    R = 3956.0 if miles else 6371.0

    # Convert all pandas Series to NumPy arrays in radians
    phi1 = np.radians(lat1_series.to_numpy())
    phi2 = np.radians(lat2_series.to_numpy())
    delta_phi = np.radians((lat2_series - lat1_series).to_numpy())
    delta_lambda = np.radians((lon2_series - lon1_series).to_numpy())

    # Haversine formula component calculations
    a = np.sin(delta_phi / 2.0)**2 + \
        np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2.0)**2

    # Compute the arc distance
    c = 2.0 * np.arcsin(np.sqrt(a))

    # Return as a pandas Series preserving the original index
    return pd.Series(R * c, index=lat1_series.index)

In [ ]:
train_df['distance_km']=calculate_series_distance(train_df['Restaurant_latitude'],train_df['Restaurant_longitude'],train_df['Delivery_location_latitude'],train_df['Delivery_location_longitude'])

In [ ]:
train_df.head()

In [ ]:
train_df.info()

---
## 7. Feature selection

ID-type columns (`ID`, `Delivery_person_ID`), the raw lat/long columns (already
captured by `distance_km`) and the raw time columns (already captured by
`order_hour`) are dropped by simply not selecting them.

In [ ]:
features=['Delivery_person_Age','Delivery_person_Ratings','distance_km','order_hour','Weatherconditions','Road_traffic_density',
          'Vehicle_condition','Type_of_order','Type_of_vehicle','multiple_deliveries','Festival','City']

In [ ]:
X=train_df[features]
y=train_df['Time_taken(min)']

In [ ]:
X.info()

In [ ]:
X.shape

In [ ]:
y.shape

---
## 8. Outlier handling + per-feature EDA

### 8.1 `distance_km` — outlier removal

A handful of rows have distances in the thousands of km, which is impossible for a
food delivery. These come from corrupted / sign-flipped lat-long values. Rows above
100 km are dropped.

In [ ]:
X['distance_km'].describe()

In [ ]:
X['distance_km'].sort_values().tail(100)

In [ ]:
# The bad rows: distances way beyond any real food delivery
X[X['distance_km']>100]['distance_km']

In [ ]:
sns.histplot(X['distance_km'],bins=100)

In [ ]:
# Inspecting the raw rows behind those distances -> broken lat/long values
train_df[train_df['distance_km']>100]

In [ ]:
# Drop the impossible distances. reset_index on BOTH X and y so they stay aligned.
mask=X['distance_km']<100
X=X[mask].reset_index(drop=True)
y=y[mask].reset_index(drop=True)

In [ ]:
X['distance_km'].describe()

In [ ]:
sns.histplot(X['distance_km'],bins=10)

In [ ]:
# Correlation of distance with delivery time -> ~0.32 (positive but not the whole story,
# which is why the tree models beat linear regression later)
X["distance_km"].corr(y)

### 8.2 `Delivery_person_Age`

In [ ]:
X['Delivery_person_Age'].describe()

In [ ]:
sns.histplot(X['Delivery_person_Age'],bins=10)

### 8.3 `Delivery_person_Ratings` — capping

Ratings are on a 1–5 scale but some rows contain values above 5 (data entry noise),
so everything is clipped at 5.

In [ ]:
X['Delivery_person_Ratings'].describe()

In [ ]:
X["Delivery_person_Ratings"]=X["Delivery_person_Ratings"].clip(upper=5)

In [ ]:
sns.histplot(X['Delivery_person_Ratings'],bins=10)

### 8.4 `Weatherconditions` vs delivery time

Group-by check to confirm weather actually matters before encoding it.

Result from the original run — Sunny is clearly the fastest (~21.9 min mean) while
Cloudy/Fog are the slowest (~28.9 min). So the feature is worth keeping.

In [ ]:
X['Weatherconditions'].value_counts()

In [ ]:
temp=X.copy()
temp["time_taken"]=y.copy()
temp.groupby('Weatherconditions')['time_taken'].agg(["count",'mean',"median","std"])

### 8.5 `Road_traffic_density` vs delivery time

Clear monotonic ordering in the original output — Low 21.4 → Medium 26.7 →
High 27.2 → Jam 31.2 min. Because the categories have a natural order, this one gets
**ordinal encoding** (Section 9) rather than one-hot.

In [ ]:
X['Road_traffic_density'].value_counts()

In [ ]:
temp=X.copy()
temp["time_taken"]=y.copy()
temp.groupby('Road_traffic_density')['time_taken'].agg(["count",'mean',"median","std"])

### 8.6 `order_hour` vs delivery time  *(Part 4 experiment)*

The group-by shows a clear rush-hour pattern: ~19.5 min in the morning hours (8–10),
jumping to ~26–27 min at lunch (11–14) and again in the evening (17+).

An `order_period` feature (Lunch / Dinner / Normal) was built to capture that.
**Note:** in the final Part 4 run, both `order_hour` and `order_period` were dropped
before modelling — that is why the final feature list has 20 columns and no hour
feature. The code is kept here as the experiment that was run.

In [ ]:
temp=X.copy()
temp["time_taken"]=y.copy()
temp.groupby('order_hour')['time_taken'].agg(["count",'mean',"median","std"])

In [ ]:
# Bucket the raw hour into meal periods
def order_period(hour):
  if 11<=hour<=14 :
    return "Lunch"
  elif 17<=hour<=21 :
    return "Dinner"
  else :
    return "Normal"

In [ ]:
X['order_hour'].apply(order_period)

In [ ]:
X['order_period']=X['order_hour'].apply(order_period)

In [ ]:
# Final decision in Part 4: drop BOTH the raw hour and the derived period.
# (Comment out this cell if you want to keep the time features — you would then need to
#  one-hot encode 'order_period' along with the other categoricals in Section 9.)
X.drop(columns=['order_hour','order_period'],inplace=True)

In [ ]:
X.info()

### 8.7 `Festival`, `City`, `Type_of_vehicle`, `Type_of_order`

The festival effect is huge in the original output: **25.9 min normally vs 45.5 min
during a festival** — the single strongest categorical signal in the dataset.

In [ ]:
X['City'].value_counts()

In [ ]:
X['Festival'].value_counts()

In [ ]:
temp=X.copy()
temp["time_taken"]=y.copy()
temp.groupby('Festival')['time_taken'].agg(["count",'mean',"median","std"])

In [ ]:
X['Type_of_vehicle'].value_counts()

In [ ]:
X['Type_of_order'].value_counts()

---
## 9. Encoding categorical features

Two different strategies, chosen on purpose:

* **Ordinal encoding** for `Road_traffic_density` — Low < Medium < High < Jam is a
  real ordering, so mapping it to 0/1/2/3 preserves information a one-hot would throw away.
* **One-hot encoding** for everything else (`Weatherconditions`, `Type_of_order`,
  `Type_of_vehicle`, `Festival`, `City`) — no natural order between the categories.
  `drop_first=True` avoids the dummy variable trap (perfect multicollinearity), which
  matters for Linear Regression.

In [ ]:
# One-hot: weather has no natural ordering
X=pd.get_dummies(X,columns=['Weatherconditions'],drop_first=True,dtype=int)

In [ ]:
X.info()

In [ ]:
# Ordinal: traffic density DOES have a natural ordering
traffic_mapping={
    'Low':0,
    'Medium': 1,
    'High' : 2,
    'Jam' : 3
}

X["Road_traffic_density"]=X["Road_traffic_density"].map(traffic_mapping)

In [ ]:
X.info()

In [ ]:
# One-hot the remaining nominal categoricals
X=pd.get_dummies(X,columns=['Type_of_order','Type_of_vehicle','Festival','City'],drop_first=True,dtype=int)

In [ ]:
# Final feature matrix: 20 numeric columns, ready for modelling
X.info()

---
## 10. Train/test split + scaling

* 80/20 split, `random_state=42` so results are reproducible.
* `StandardScaler` is **fit on the training set only** and then applied to the test set
  — fitting on the full data would leak test information into training.
* The scaled arrays are wrapped back into DataFrames so the column names survive
  (needed later to read the feature importances).
* Scaling is essential for Linear Regression; the tree models don't need it, but it is
  applied uniformly here so every model sees identical input.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler= StandardScaler()

In [ ]:
# fit_transform on train, transform ONLY on test -> no data leakage
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [ ]:
# Put the column names back so feature importances stay readable
X_train_scaled=pd.DataFrame(X_train_scaled,columns=X_train.columns,index=X_train.index)
X_test_scaled=pd.DataFrame(X_test_scaled,columns=X_test.columns,index=X_test.index)

---
## 11. Model training

Four regressors, trained on identical data, compared on RMSE and R² on the held-out
test set. Recorded results from the original Part 4 run are noted above each block.

### 11.1 Linear Regression — baseline

**Result: RMSE 6.22, R² 0.568**

The baseline. It only captures linear relationships, so it misses the interaction
effects (e.g. traffic × distance × festival) that drive delivery time.

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lr=LinearRegression()

In [ ]:
lr.fit(X_train_scaled,y_train)

In [ ]:
lr.intercept_

In [ ]:
# Coefficients line up positionally with X_train.columns below
lr.coef_

In [ ]:
X_train.columns

In [ ]:
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
mse=mean_squared_error(y_test,lr.predict(X_test_scaled))
rmse=np.sqrt(mse)
r2=r2_score(y_test,lr.predict(X_test_scaled))
print("rmse",rmse)
print("r2",r2)
# Original run -> rmse 6.2216 | r2 0.5681

### 11.2 Decision Tree — captures non-linearity, but overfits

**Result: test RMSE 5.46, R² 0.667  |  train RMSE 0.04, R² 0.99998**

Better than linear regression on test, but look at the train-vs-test gap: the tree
memorises the training set almost perfectly (R² ≈ 1.0) and generalises far worse.
That is textbook overfitting, and it is exactly the reason for moving to ensembles
(Random Forest / XGBoost) next.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

In [ ]:
# No max_depth -> the tree grows until leaves are pure, which is what causes the overfit
dt=DecisionTreeRegressor(random_state=42)

In [ ]:
dt.fit(X_train_scaled,y_train)

In [ ]:
y_pred_dt=dt.predict(X_test_scaled)

In [ ]:
# TEST performance
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
mse=mean_squared_error(y_test,dt.predict(X_test_scaled))
rmse=np.sqrt(mse)
r2=r2_score(y_test,dt.predict(X_test_scaled))
print("rmse",rmse)
print("r2",r2)
# Original run -> rmse 5.4592 | r2 0.6674

In [ ]:
# TRAIN performance -> near-perfect, proving the overfit
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
mse=mean_squared_error(y_train,dt.predict(X_train_scaled))
rmse=np.sqrt(mse)
r2=r2_score(y_train,dt.predict(X_train_scaled))
print("rmse",rmse)
print("r2",r2)
# Original run -> rmse 0.0401 | r2 0.99998  <-- overfitting

### 11.3 Random Forest — bagging fixes the overfit

**Result: RMSE 4.06, R² 0.816**

Averaging 100 de-correlated trees cancels out the variance that made the single tree
overfit. Big jump over both previous models.

Top features by importance in the original run:
`Delivery_person_Ratings` (0.21) > `distance_km` (0.147) > `Road_traffic_density` (0.127) >
`Vehicle_condition` (0.121) > `Delivery_person_Age` (0.106).

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
rf=RandomForestRegressor(n_estimators=100,random_state=42)

In [ ]:
rf.fit(X_train_scaled,y_train)

In [ ]:
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
mse=mean_squared_error(y_test,rf.predict(X_test_scaled))
rmse=np.sqrt(mse)
r2=r2_score(y_test,rf.predict(X_test_scaled))
print("rmse",rmse)
print("r2",r2)
# Original run -> rmse 4.0564 | r2 0.8164

In [ ]:
rf.feature_importances_

In [ ]:
# Read the importances against these names (same positional order)
X_train.columns

In [ ]:
# Readable version of the same thing
pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

### 11.4 XGBoost — best model

**Result: RMSE 3.99, R² 0.823**  ← best of the four

Gradient boosting builds trees sequentially, each correcting the previous one's
errors, so it edges out the Random Forest.

XGBoost weights the features quite differently from RF — it leans much harder on
`multiple_deliveries` (0.160), `Vehicle_condition` (0.132), `Road_traffic_density` (0.126)
and `Festival_Yes` (0.109).

In [ ]:
from xgboost import XGBRegressor

In [ ]:
xgb=XGBRegressor(n_estimators=100,random_state=42)

In [ ]:
xgb.fit(X_train_scaled,y_train)

In [ ]:
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
mse=mean_squared_error(y_test,xgb.predict(X_test_scaled))
rmse=np.sqrt(mse)
r2=r2_score(y_test,xgb.predict(X_test_scaled))
print("rmse",rmse)
print("r2",r2)
# Original run -> rmse 3.9880 | r2 0.8225  <-- BEST

In [ ]:
xgb.feature_importances_

In [ ]:
X_train.columns

In [ ]:
pd.Series(xgb.feature_importances_, index=X_train.columns).sort_values(ascending=False)

---
## 12. Results comparison + conclusion

| Model | Test RMSE | Test R² | Note |
|---|---|---|---|
| Linear Regression | 6.22 | 0.568 | Baseline; can't model interactions |
| Decision Tree | 5.46 | 0.667 | Overfits badly (train R² 0.99998) |
| Random Forest | 4.06 | 0.816 | Bagging removes the variance problem |
| **XGBoost** | **3.99** | **0.823** | **Best model** |

**Conclusion:** XGBoost is the best performer, predicting delivery time within roughly
±4 minutes on unseen orders and explaining ~82% of the variance. The jump from the
single Decision Tree (0.667) to the ensembles (0.816 / 0.823) is the headline result —
it shows the problem is strongly non-linear *and* that a single unregularised tree
cannot generalise.

**What drives delivery time**, combining both models' importances: delivery person
rating and age, distance, road traffic density, vehicle condition, number of
simultaneous deliveries, and whether it's a festival day.

**Possible next steps:** hyperparameter tuning (`GridSearchCV` / `RandomizedSearchCV`)
on XGBoost, cross-validation instead of a single split, capping the Decision Tree's
`max_depth` for a fair comparison, and re-testing the `order_period` feature that was
dropped in Section 8.6.

In [ ]:
# Quick side-by-side of all four models
results = pd.DataFrame({
    "Model": ["Linear Regression", "Decision Tree", "Random Forest", "XGBoost"],
    "RMSE":  [np.sqrt(mean_squared_error(y_test, m.predict(X_test_scaled))) for m in [lr, dt, rf, xgb]],
    "R2":    [r2_score(y_test, m.predict(X_test_scaled))                    for m in [lr, dt, rf, xgb]],
}).sort_values("RMSE").reset_index(drop=True)

results

In [ ]:
# Visual comparison
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(results["Model"], results["RMSE"], color="steelblue")
ax[0].set_title("RMSE (lower is better)"); ax[0].tick_params(axis='x', rotation=30)
ax[1].bar(results["Model"], results["R2"], color="seagreen")
ax[1].set_title("R2 (higher is better)");  ax[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()